In [1]:
%load_ext autoreload
%autoreload 2

In [25]:
import numpy as np
import random

import pickle as pkl
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainwidemap import bwm_query, bwm_units, load_good_units, load_trials_and_mask
from tqdm import tqdm
from one.api import ONE
from brainbox.singlecell import bin_spikes2D
from iblatlas.regions import BrainRegions
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [10]:
from ibl_info.manifold import get_psth, process_single_session, get_trial_masks, get_target_pids
from ibl_info.utils import check_config

In [91]:
config = check_config()
MY_REGIONS = config["stim_prior_regions"]
MIN_NEURONS = config["min_units"]
BIN_SIZE = 0.0125
STRIDE = 0.002
USE_SLIDING_WINDOW = True

EPOCHS = {
    "Quiescent": {"align": "stimOn_times", "window": (0.6, 0.0)},
    "Stimulus": {"align": "stimOn_times", "window": (0.0, 0.15)},
    "Choice": {"align": "firstMovement_times", "window": (0.15, 0.0)},
}

In [5]:
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
)

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"


In [ ]:
relevant_pids = get_target_pids(one, MY_REGIONS)

bwm_df = bwm_query(one)
subset_df = bwm_df[bwm_df["pid"].isin(relevant_pids)]

In [127]:
one = ONE()
print("Querying BWM Units...")

units_df = bwm_units(one)
relevant_pids = units_df[units_df["Beryl"].isin(MY_REGIONS)]["pid"].unique()

bwm_df = bwm_query(one)
subset_df = bwm_df[bwm_df["pid"].isin(relevant_pids)]

Querying BWM Units...
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
Loading bwm_query results from fixtures/2023_12_bwm_release.csv


In [128]:
subset_df

,pid,eid,probe_name,session_number,date,subject,lab
0,56f2a378-78d2-4132-b3c8-8c1ba82be598,6713a4a7-faed-4df2-acab-ee4e63326f8d,probe00,1,2020-02-18,NYU-11,angelakilab
2,6be21156-33b0-4f70-9a0f-65b3e3cd6d4a,56956777-dca5-468c-87cb-78150432cc57,probe00,1,2020-02-21,NYU-11,angelakilab
4,c893c0a3-5597-49cf-baa1-60efdfdef542,b182b754-3c3e-4942-8144-6ee790926b58,probe01,1,2020-01-21,NYU-12,angelakilab
5,1e176f17-d00f-49bb-87ff-26d237b525f1,a8a8af78-16de-4841-ab07-fde4b5281a03,probe00,1,2020-01-22,NYU-12,angelakilab
8,d213e786-4b1c-477d-a710-766d69fa1ac1,032ffcdf-7692-40b3-b9ff-8def1fc18b2e,probe00,1,2020-01-23,NYU-12,angelakilab
...,...,...,...,...,...,...,...
692,d14f70e6-bf7b-4d6d-a380-bfd0a46ed7a1,5522ac4b-0e41-4c53-836a-aaa17e82b9eb,probe01,1,2020-09-15,CSH_ZAD_029,zadorlab
693,81b1800e-f138-4337-abc9-bb0449b77852,993c7024-0abc-4028-ad30-d397ad55b084,probe00,1,2020-09-16,CSH_ZAD_029,zadorlab
694,8bf0f1a4-0d8c-4df3-a99e-f7c81c809652,993c7024-0abc-4028-ad30-d397ad55b084,probe01,1,2020-09-16,CSH_ZAD_029,zadorlab
695,5d570bf6-a4c6-4bf1-a14b-2c878c84ef0e,fece187f-b47f-4870-a1d6-619afe942a7d,probe01,1,2020-09-17,CSH_ZAD_029,zadorlab


In [21]:
pid = subset_df["pid"][0]
eid = subset_df["eid"][0]

In [110]:
session_results = process_single_session(
    pid,
    eid,
    MY_REGIONS,
    EPOCHS,
)

In [116]:
q = get_psth(
    region_spike_times,
    region_spike_ids,
    target_ids,
    align_times,
    params["window"][0],
    params["window"][1],
)

In [117]:
q.shape

(135, 12, 48)